# OneBox vs Blackrock: before and after synchronization

The sync pulses and the behaviour signals (eye-x, eye-y, pupil, photodiode) are recorded
twice, by the OneBox and by Blackrock, each on its own clock.

1. **Before**: a few minutes of the raw files, each on its own clock (seconds from its
   own start). The two copies do not line up.
2. **After**: the same minutes as the pipeline exported them: the OneBox
   `export/*.analog.mat` is stamped with Blackrock time (`t0_blackrock`,
   `fs_blackrock`), and Blackrock's samples carry their own NSP (PTP) timestamps. The
   two copies line up.
3. **Numbers**: `export/sync_report.json`, written by `validate_remapping`.

Nothing is fitted or transformed here. Raw files are read directly: the OneBox `.obx.bin`
with `np.memmap`, Blackrock's `.ns5`/`.ns2` with neo (parsing a `.ns5` over the share
takes a few minutes).

In [ ]:
import json
import sys
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from neo.rawio import BlackrockRawIO

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
import spikesorting as ss

# ---------------- edit these ----------------
SESSION = "session_template_1probe"          # the session config's name
MACHINE = {"darwin": "mac", "win32": "windows_rig"}.get(sys.platform, "hpc")
START_MIN = 60.0                             # window start, minutes into the recording
DURATION_MIN = 2.0                           # window length, minutes
ZOOM_MS = 20.0                               # half-width of the zoom on one 1 Hz edge
MAX_POINTS = 200_000                         # per trace drawn

## 1. Open the files

The OneBox `.obx.bin` is int16 with `nSavedChans` columns: row *n* is XA*n* (all
channels saved), the last column is the SY word whose bit 6 is the 1 Hz. Volts per count
is `obAiRangeMax / obMaxInt`. Blackrock's sync channels are in the `.ns5`, its behaviour
copies in the `.ns2` at the positions `blackrock.analog_channels` names.

In [ ]:
config = ss.load_session_config(SESSION, MACHINE)
_, npx = config.per_probe()[0]              # one OneBox per run: it sits with the first probe
brk, nps = config.blackrock, npx.neuropixels

# OneBox raw: memory-map the .obx.bin
obx_bin = sorted(Path(nps.bin_file).parents[1].glob("*.obx.bin"))[0]
meta = dict(line.split("=", 1) for line in obx_bin.with_suffix(".meta").read_text().splitlines()
            if "=" in line)
assert meta["snsSaveChanSubset"] == "all", "a channel subset was saved: row n is not XA n"
obx_fs = float(meta["obSampRate"])
obx_v = float(meta["obAiRangeMax"]) / float(meta["obMaxInt"])
obx = np.memmap(obx_bin, dtype=np.int16, mode="r").reshape(-1, int(meta["nSavedChans"]))
print(f"{obx_bin.name}: {obx.shape[0] / obx_fs / 60:.1f} min, {obx.shape[1]} channels, {obx_fs:g} Hz")

# OneBox synchronized: the export, stamped with Blackrock time
mat_path = sorted(npx.paths.export_for("neuropixels").glob("*.analog.mat"))[0]
mat = h5py.File(mat_path, "r")["analog"]
t0_br, fs_br = float(mat["t0_blackrock"][()].ravel()[0]), float(mat["fs_blackrock"][()].ravel()[0])
print(f"{mat_path.name}: sample k is at {t0_br:.6f} + k / {fs_br:.6f} s (Blackrock time)")

# Blackrock raw: neo, one reader per file
readers = {}
for path in (brk.sync_file, brk.analog_file):
    nsx = int(path.suffix[3:])               # .ns5 -> 5
    readers[nsx] = BlackrockRawIO(filename=str(path.with_suffix("")), nsx_to_load=nsx,
                                  load_nev=False, gap_tolerance_ms=brk.gap_tolerance_ms)
    readers[nsx].parse_header()
    print(f"{path.name}: {readers[nsx].get_signal_sampling_rate(0):g} Hz, channels "
          f"{list(readers[nsx].header['signal_channels']['name'])}")

# label -> (OneBox column, Blackrock file, Blackrock channel index)
ids5 = list(readers[5].header["signal_channels"]["id"])


def ns5_index(channel):
    """A .ns5 channel named by its id, else by its 0-based position (the pipeline's rule)."""
    return ids5.index(str(channel)) if str(channel) in ids5 else channel


channels = {
    "1 Hz": (obx.shape[1] - 1, 5, ns5_index(brk.sync_1hz_channel)),
    "sync_burst": (nps.burst_word, 5, ns5_index(brk.burst_channel)),
    **{name: (nps.analog_channels[name], 2, position)
       for name, position in brk.analog_channels.items() if name in nps.analog_channels},
}
for name, (col, nsx, index) in channels.items():
    br_name = (readers[nsx].header["signal_channels"]["name"][index]
               if index < len(readers[nsx].header["signal_channels"]) else "missing")
    print(f"  {name:<11} OneBox column {col:>2}  |  .ns{nsx} {br_name}")

## 2. Before synchronization: each system on its own clock

`START_MIN` to `START_MIN + DURATION_MIN` minutes into **each** recording, timed from
that recording's own first sample. Blackrock solid black, OneBox dashed orange, both
in volts on one axis. Only the 1 Hz is scaled 0–1: the OneBox copy is a bit of the SY word,
Blackrock's a voltage.

In [ ]:
def read_onebox_raw(name, col, k0, k1):
    """OneBox raw samples k0:k1 of one column: the SY word's bit 6 for the 1 Hz, else volts."""
    y = obx[k0:k1, col]
    return ((y.astype(np.int64) >> 6) & 1).astype(np.float64) if name == "1 Hz" else y * obx_v


def read_blackrock(nsx, index, s0, s1):
    """Blackrock samples s0:s1 of one channel, in volts (neo gives mV for the ainp inputs)."""
    reader = readers[nsx]
    raw = reader.get_analogsignal_chunk(0, 0, s0, s1, stream_index=0, channel_indexes=[index])
    y = reader.rescale_signal_raw_to_float(raw, dtype="float64", stream_index=0,
                                           channel_indexes=[index]).ravel()
    return y / {"V": 1.0, "mV": 1e3, "uV": 1e6}[str(reader.header["signal_channels"]["units"][index])]


def scaled_01(sides):
    """The 1 Hz is a bit on the OneBox and a voltage on Blackrock: put both on 0-1."""
    return {side: (t, (y - y.min()) / (np.ptp(y) or 1) if y.size else y) for side, (t, y) in sides.items()}


def plot_panels(traces, origin, scale, xlabel, title, lo=-np.inf, hi=np.inf):
    """One panel per label, Blackrock solid black over OneBox dashed orange, x = (t - origin) / scale,
    drawn for lo <= t <= hi."""
    fig, axes = plt.subplots(len(traces), 1, sharex=True, figsize=(13, 1.4 * len(traces) + 1))
    for ax, (name, sides) in zip(axes, traces.items()):
        for side, style in (("Blackrock", dict(color="black")), ("OneBox", dict(color="tab:orange", ls="--"))):
            if side not in sides:
                continue
            t, y = sides[side]
            keep = (t >= lo) & (t <= hi)
            step = max(1, keep.sum() // MAX_POINTS)
            ax.plot((t[keep][::step] - origin) / scale, y[keep][::step], lw=0.8, label=side, **style)
        ax.set_ylabel(name if name == "1 Hz" else f"{name} (V)", rotation=0, ha="right", va="center")
    axes[0].legend(loc="upper right", fontsize=8, ncol=2)
    axes[-1].set_xlabel(xlabel)
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


k0, k1 = int(START_MIN * 60 * obx_fs), int((START_MIN + DURATION_MIN) * 60 * obx_fs)
before = {}                                  # label -> {"OneBox"/"Blackrock": (seconds, values)}
for name, (col, nsx, index) in channels.items():
    before[name] = {"OneBox": (np.arange(k0, k1) / obx_fs, read_onebox_raw(name, col, k0, k1))}
    fs = readers[nsx].get_signal_sampling_rate(0)
    if index < len(readers[nsx].header["signal_channels"]):
        s0, s1 = int(START_MIN * 60 * fs), int((START_MIN + DURATION_MIN) * 60 * fs)
        before[name]["Blackrock"] = (np.arange(s0, s1) / fs, read_blackrock(nsx, index, s0, s1))
before["1 Hz"] = scaled_01(before["1 Hz"])

plot_panels(before, origin=0.0, scale=60, xlabel="minutes from each recording's own start",
            title=f"before synchronization: minutes {START_MIN:g}-{START_MIN + DURATION_MIN:g} of each recording")

## 3. After synchronization: both on Blackrock time

The same OneBox samples, now read from the export and timed by its stamp
(`t0_blackrock + k / fs_blackrock`); the 1 Hz is not in the export, so it is the raw
SY bit on the same stamp. Blackrock's samples over the same stretch of time, each at its
own PTP timestamp. The x-axis is minutes into the OneBox recording, so the window is
the one above.

In [ ]:
t_obx = t0_br + np.arange(k0, k1) / fs_br           # the export's stamp
mat_names = "".join(chr(c) for c in mat["names"][()].ravel()).split(",")
print(f"OneBox samples {k0}-{k1}: minutes {START_MIN:g}-{START_MIN + DURATION_MIN:g} of the OneBox, "
      f"Blackrock time {t_obx[0]:.3f}-{t_obx[-1]:.3f} s")

after = {}
for name, (col, nsx, index) in channels.items():
    if name == "1 Hz":                      # not in the export: the raw SY bit, on the export's stamp
        y = read_onebox_raw(name, col, k0, k1)
    else:                                   # from the export: row index.<name>, volts per digit
        row = int(mat["index"][name][()].ravel()[0]) - 1 if name in mat_names else col
        y = mat["data"][k0:k1, row] * float(mat["v_per_digit"][()].ravel()[row])
    after[name] = {"OneBox": (t_obx, y)}

    if index < len(readers[nsx].header["signal_channels"]):
        stamps = readers[nsx]._nsx_data_header[nsx][0]["timestamp"]           # PTP, per sample
        resolution = float(readers[nsx]._nsx_basic_header[nsx]["timestamp_resolution"])
        s0, s1 = np.searchsorted(stamps, [t_obx[0] * resolution, t_obx[-1] * resolution])
        after[name]["Blackrock"] = (stamps[s0:s1] / resolution, read_blackrock(nsx, index, s0, s1))
after["1 Hz"] = scaled_01(after["1 Hz"])

# One 1 Hz rising edge in the window, for the zoom.
rising = np.flatnonzero(np.diff(after["1 Hz"]["OneBox"][1]) > 0)
t_edge = t_obx[rising[0] + 1] if rising.size else t_obx[t_obx.size // 2]

plot_panels(after, origin=t_obx[0] - START_MIN * 60, scale=60, xlabel="minutes into the OneBox recording",
            title=f"after synchronization: minutes {START_MIN:g}-{START_MIN + DURATION_MIN:g}")
plot_panels(after, origin=t_edge, scale=1e-3, xlabel="ms from the OneBox 1 Hz edge (Blackrock time)",
            title=f"after synchronization: +/-{ZOOM_MS:g} ms around one 1 Hz edge",
            lo=t_edge - ZOOM_MS / 1e3, hi=t_edge + ZOOM_MS / 1e3)

## 3b. Every 1 Hz edge: how much the alignment varies

The zoom above is one edge. Here it is every edge of the session. The OneBox 1 Hz
edges (`npx_obx_1hz.txt`, OneBox seconds) go through the **same stamp the analog
export carries** (`blackrock_slope * t + blackrock_intercept`), and each is paired with
the nearest Blackrock 1 Hz edge (`blackrock_1hz.txt`, NSP seconds). The difference,
Blackrock minus OneBox, is the alignment error of the analog export at that second.

Read from the edge files `extract_sync` wrote, so no recording is opened. Not held
out: the 1 Hz trains are what the map was fitted on (the burst is the held-out check,
section 4). What this shows is the **spread**: how far a single linear map is from
each edge across the session, and whether it wanders (residual vs time) or is noise
(flat band). Edges more than `PAIR_MS` apart are counted as unpaired, not plotted.

In [ ]:
PAIR_MS = 250.0                                   # farther apart than this: not the same pulse

obx_edges = np.loadtxt(npx.paths.sync_for("neuropixels") / "npx_obx_1hz.txt", ndmin=1)
br_edges = np.loadtxt(config.paths.sync_for("blackrock") / "blackrock_1hz.txt", ndmin=1)
slope = float(mat["blackrock_slope"][()].ravel()[0])
intercept = float(mat["blackrock_intercept"][()].ravel()[0])
mapped = slope * obx_edges + intercept           # OneBox edges on Blackrock time

# Nearest Blackrock edge to each mapped OneBox edge.
j = np.clip(np.searchsorted(br_edges, mapped), 1, br_edges.size - 1)
j -= (mapped - br_edges[j - 1]) < (br_edges[j] - mapped)
residual_ms = (br_edges[j] - mapped) * 1e3
paired = np.abs(residual_ms) < PAIR_MS
res, t_min = residual_ms[paired], obx_edges[paired] / 60

print(f"{paired.sum()} of {obx_edges.size} OneBox edges paired ({(~paired).sum()} unpaired), "
      f"{br_edges.size} Blackrock edges")
print(f"Blackrock - OneBox: median {np.median(res):+.3f} ms, SD {res.std():.3f} ms, "
      f"|99th pct| {np.percentile(np.abs(res - np.median(res)), 99):.3f} ms, "
      f"range {res.min():+.3f} to {res.max():+.3f} ms")

fig, (ax_t, ax_h) = plt.subplots(1, 2, figsize=(13, 3.6), gridspec_kw={"width_ratios": [3, 1]},
                                 sharey=True)
ax_t.plot(t_min, res, ".", ms=1.5, color="tab:blue")
ax_t.axhline(0, color="k", lw=0.5)
ax_t.set_xlabel("minutes into the OneBox recording")
ax_t.set_ylabel("Blackrock - OneBox edge (ms)")
ax_t.set_title("every 1 Hz edge, OneBox mapped by the analog export's stamp")
ax_h.hist(res, bins=100, orientation="horizontal", color="tab:blue")
ax_h.set_xlabel("edges")
fig.tight_layout()
plt.show()

## 4. What the export says

`sync_report.json`, written by `validate_remapping`: the verdict, what picked the
whole-second shift (the 14 s burst, or a channel standing in for it), each signal that
fitted or tested the map, and Blackrock's own copies checked against the OneBox (a
constant offset is Blackrock's filter; the **drift** is what synchronization is judged
on).

In [ ]:
report = json.loads((npx.paths.export_for("neuropixels") / "sync_report.json").read_text())
print(f"verdict: {report['verdict']}")
print(report["verdict_note"])
display(pd.DataFrame(report["signals"]).set_index("signal").drop(columns=["vote"], errors="ignore"))
display(pd.DataFrame(report.get("independent_checks") or []).set_index("channel"))